# Financial PhraseBank 75Agree: загрузка датасета и EDA через YData Profiling

В этом ноутбуке мы:

1. Скачиваем оригинальный архив Financial PhraseBank.
2. Извлекаем файл `Sentences_75Agree.txt`.
3. Преобразуем его в таблицу `sentence, label`.
4. Сохраняем CSV для дальнейшего обучения ML-модели.
5. Используем open-source библиотеку `ydata-profiling` для автоматического EDA-отчета.

Версия `75Agree` выбрана потому, что в нее входят только предложения, где более 75% аннотаторов согласились по sentiment label. Это уменьшает шум в разметке.

## 1. Установка зависимостей

Если пакеты уже установлены, эта ячейка ничего лишнего не делает. Библиотека устанавливается как `ydata-profiling`, а импортируется как `ydata_profiling`.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "pandas": "pandas",
    "requests": "requests",
    "certifi": "certifi",
    "ydata_profiling": "ydata-profiling",
}

missing_packages = [
    pip_name
    for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("All required packages are already installed.")

## 2. Скачивание Financial PhraseBank

Источник: Hugging Face mirror исходного архива `FinancialPhraseBank-v1.0.zip`. Внутри архива есть четыре версии датасета: `50Agree`, `66Agree`, `75Agree`, `AllAgree`. Для проекта используем `Sentences_75Agree.txt`.

In [ ]:
from pathlib import Path
import zipfile

import certifi
import pandas as pd
import requests

DATA_URL = "https://huggingface.co/datasets/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip"
DATA_DIR = Path("data")
REPORT_DIR = Path("reports")
DATA_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

zip_path = DATA_DIR / "FinancialPhraseBank-v1.0.zip"

if not zip_path.exists():
    response = requests.get(DATA_URL, timeout=90, verify=certifi.where())
    response.raise_for_status()
    zip_path.write_bytes(response.content)
    print(f"Downloaded archive to {zip_path.resolve()}")
else:
    print(f"Archive already exists: {zip_path.resolve()}")

print(f"Archive size: {zip_path.stat().st_size / 1024:.1f} KB")

## 3. Извлечение `75Agree` и преобразование в CSV

Оригинальный формат файла:

```text
sentence@sentiment
```

В коде используем `rsplit('@', 1)`, чтобы корректно разделять строку по последнему символу `@`.

In [ ]:
TARGET_FILE = "FinancialPhraseBank-v1.0/Sentences_75Agree.txt"

with zipfile.ZipFile(zip_path) as archive:
    raw_text = archive.read(TARGET_FILE).decode("latin-1")

rows = []
for line in raw_text.splitlines():
    line = line.strip()
    if not line:
        continue
    sentence, label = line.rsplit("@", 1)
    rows.append({
        "sentence": sentence.strip(),
        "label": label.strip().lower(),
    })

df = pd.DataFrame(rows)

label_order = ["negative", "neutral", "positive"]
if sorted(df["label"].unique()) != sorted(label_order):
    raise ValueError(f"Unexpected labels: {sorted(df['label'].unique())}")

csv_path = DATA_DIR / "financial_phrasebank_75agree.csv"
df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Saved CSV to {csv_path.resolve()}")
print(f"Rows: {len(df)}")
df.head()

## 4. Быстрая проверка структуры датасета

Проверяем размер, пропуски, дубликаты и распределение классов. Это пригодится для презентации и для следующего этапа с обучением модели.

In [ ]:
print("Shape:", df.shape)
print("
Missing values:")
print(df.isna().sum())
print("
Duplicate rows:", df.duplicated().sum())

class_distribution = (
    df["label"]
    .value_counts()
    .reindex(label_order)
    .rename_axis("label")
    .reset_index(name="count")
)
class_distribution["share"] = class_distribution["count"] / len(df)
class_distribution

In [ ]:
ax = (
    df["label"]
    .value_counts()
    .reindex(label_order)
    .plot(kind="bar", title="Financial PhraseBank 75Agree: class distribution")
)
ax.set_xlabel("Sentiment label")
ax.set_ylabel("Number of sentences");

## 5. Автоматический EDA-отчет через YData Profiling

`YData Profiling` генерирует HTML-отчет с обзором датасета: типы колонок, пропуски, распределения, предупреждения по качеству данных и базовую статистику по текстовой колонке.

Отчет сохраняется в папку `reports/`.

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="Financial PhraseBank 75Agree - YData Profiling Report",
    explorative=True,
)

report_path = REPORT_DIR / "financial_phrasebank_75agree_ydata_report.html"
profile.to_file(report_path)

print(f"YData Profiling report saved to: {report_path.resolve()}")

## 6. Отображение отчета внутри ноутбука

Если отчет слишком долго открывается внутри Jupyter, можно открыть HTML-файл из папки `reports/` в браузере.

In [ ]:
profile.to_notebook_iframe()

## 7. Вывод для проекта

Для дальнейшего обучения модели используем файл:

```text
data/financial_phrasebank_75agree.csv
```

Целевая переменная:

```text
label ∈ {negative, neutral, positive}
```

Следующий этап проекта: обучить baseline-модель `TF-IDF + Logistic Regression`, затем сравнить качество по `accuracy`, `macro F1-score`, `weighted F1-score` и `recall` для негативного класса.